# BeamDeflection - Jupyter Notebook

Beam deflection, bending moment, shear force and stress checks for single-span beams, converted from the **BeamDeflection Android app** (Kotlin / Jetpack Compose) to Python.

- Supports: simply supported, cantilever, fixed-fixed, propped cantilever
- Loads: point loads anywhere, full or partial uniform loads, self-weight
- Libraries: 14 materials and the IPE 100-300 catalogue
- Outputs: 2D diagrams, **3D diagrams / charts / graphs**, **comparison tables**

The solver lives in `beam_core.py` (same folder). Figures are saved next to this notebook as PNG files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3D projection)

import beam_core as bc

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 10})
pd.set_option("display.width", 140)

## 1. Example beam
IPE 200, steel S275, simply supported, span 5 m: a 20 kN point load at 2 m, a 6 kN/m uniform load and self-weight.

In [ ]:
mat = bc.material("S275")
sec = bc.ipe("IPE 200")
L = 5.0

beam = bc.GeneralInput(
    supports="simple", length_m=L, e_gpa=mat.e_gpa, i_mm4=sec.i_mm4, c_mm=sec.c_mm,
    point_loads=[bc.PointLoad(20_000, 2.0)],
    udl_n_per_m=6_000,
    self_weight_n_per_m=bc.self_weight(sec.mass_kg_per_m),
    strength_mpa=mat.strength_mpa, safety_factor=1.5, limit_denominator=360,
    shear_area_mm2=sec.shear_area_mm2, shear_factor=sec.shear_factor,
    shear_strength_mpa=mat.shear_strength_mpa,
)
res = bc.solve(beam)

summary = pd.Series({
    "Max deflection (mm)": round(res.deflection_mm, 2),
    "at x (m)": round(res.deflection_at_m, 2),
    "Span / deflection": round(res.span_ratio, 0),
    "Limit L/360 OK": res.deflection_ok,
    "Max sagging moment (kNm)": round(res.sag_moment_nm / 1e3, 2),
    "Max shear (kN)": round(res.shear_n / 1e3, 2),
    "Bending stress (MPa)": round(res.stress_mpa, 1),
    "Allowable (MPa)": round(res.allowable_mpa, 1),
    "Bending utilisation": round(res.utilisation, 2),
    "Shear utilisation": round(res.shear_utilisation, 2),
}, name="Result")
summary.to_frame()

## 2. Deflection, bending moment and shear force diagrams

In [ ]:
pr = bc.curves(beam, 401)
fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
ax[0].plot(pr.x, -pr.deflection_mm, color="#1f77b4", lw=2)
ax[0].fill_between(pr.x, -pr.deflection_mm, 0, color="#1f77b4", alpha=0.15)
ax[0].set_ylabel("Deflection (mm)"); ax[0].set_title("Deflected shape (downward negative)")
ax[1].plot(pr.x, pr.moment_nm / 1e3, color="#d62728", lw=2)
ax[1].fill_between(pr.x, pr.moment_nm / 1e3, 0, color="#d62728", alpha=0.15)
ax[1].set_ylabel("Moment (kNm)"); ax[1].set_title("Bending moment (sagging positive)")
ax[2].plot(pr.x, pr.shear_n / 1e3, color="#2ca02c", lw=2)
ax[2].fill_between(pr.x, pr.shear_n / 1e3, 0, color="#2ca02c", alpha=0.15)
ax[2].set_ylabel("Shear (kN)"); ax[2].set_xlabel("x (m)"); ax[2].set_title("Shear force")
for a in ax: a.axhline(0, color="k", lw=0.8)
plt.tight_layout(); plt.savefig("fig_2d_profiles.png"); plt.show()

## 3. 3D diagram - deformed beam coloured by bending moment
Deflection is exaggerated so the shape is visible.

In [ ]:
n = 161
xs = np.linspace(0, L, n)
p3 = bc.profile(beam, xs)
width = 0.35
ys = np.linspace(-width / 2, width / 2, 9)
X, Y = np.meshgrid(xs, ys)
k = 0.12 * L / max(p3.deflection_mm.max(), 1e-9)       # visual exaggeration factor
Z = np.tile(-p3.deflection_mm * k, (len(ys), 1))
M = np.tile(p3.moment_nm / 1e3, (len(ys), 1))
norm = Normalize(M.min(), M.max())

fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(X, Y, Z, facecolors=cm.coolwarm(norm(M)), rstride=1, cstride=1, antialiased=True)
ax.plot_surface(X, Y, np.zeros_like(Z), color="grey", alpha=0.12)      # undeformed beam
m = cm.ScalarMappable(norm=norm, cmap="coolwarm"); m.set_array([])
fig.colorbar(m, ax=ax, shrink=0.6, pad=0.08, label="Bending moment (kNm)")
ax.set_xlabel("x (m)"); ax.set_ylabel("width (m)"); ax.set_zlabel("deflection (exaggerated)")
ax.set_title("3D deformed beam (colour = bending moment)")
ax.view_init(elev=24, azim=-62); ax.set_box_aspect((3, 1, 0.8))
plt.tight_layout(); plt.savefig("fig_3d_deformed_beam.png"); plt.show()

## 4. 3D graph - deflection vs span and load
Simply supported IPE 200 / S275 under a uniform load.

In [ ]:
Ls = np.linspace(2, 8, 40)
Ws = np.linspace(1, 20, 40)             # kN/m
LL, WW = np.meshgrid(Ls, Ws)
c_ss = bc.standard_case("simple", True)
D = c_ss.defl_coeff * WW * 1e3 * LL ** 4 / (mat.e_gpa * 1e9 * sec.i_mm4 * 1e-12) * 1000.0

fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(LL, WW, D, cmap="viridis", edgecolor="none", alpha=0.95)
ax.contour(LL, WW, D, zdir="z", offset=0, cmap="viridis", levels=10)
ax.set_xlabel("span L (m)"); ax.set_ylabel("uniform load w (kN/m)"); ax.set_zlabel("max deflection (mm)")
fig.colorbar(surf, shrink=0.6, pad=0.08, label="mm")
ax.set_title("Deflection grows with L^4 and linearly with load")
ax.view_init(elev=26, azim=-125)
plt.tight_layout(); plt.savefig("fig_3d_surface_span_load.png"); plt.show()

## 5. 3D chart - support cases compared (same total load)
A total load of 20 kN, as a point load (centre, or free end for the cantilever) or spread uniformly. IPE 200, S275, L = 4 m.

In [ ]:
L4, W = 4.0, 20_000.0
sups = list(bc.SUPPORTS)
rows = []
for s in sups:
    dp, mp, vp = bc.quick_max(bc.standard_case(s, False), W, L4, mat.e_gpa, sec.i_mm4)
    du, mu, vu = bc.quick_max(bc.standard_case(s, True), W / L4, L4, mat.e_gpa, sec.i_mm4)
    rows.append((s, dp, du, mp, mu, vp, vu))
sup_df = pd.DataFrame(rows, columns=["support", "defl_point_mm", "defl_udl_mm", "M_point_Nm", "M_udl_Nm", "V_point_N", "V_udl_N"])

fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]
for i, s in enumerate(sups):
    for j, col in enumerate(["defl_point_mm", "defl_udl_mm"]):
        v = sup_df.loc[i, col]
        ax.bar3d(i, j, 0, 0.6, 0.6, v, color=colors[i], alpha=0.9 - 0.25 * j, shade=True)
        ax.text(i + 0.3, j + 0.3, v + 0.3, f"{v:.2f}", ha="center", fontsize=8)
ax.set_xticks(np.arange(4) + 0.3); ax.set_xticklabels(["simple", "cantilever", "fixed", "propped"])
ax.set_yticks([0.3, 1.3]); ax.set_yticklabels(["point", "uniform"])
ax.set_zlabel("max deflection (mm)"); ax.set_title("Max deflection by support type and load type (20 kN total)")
ax.view_init(elev=24, azim=-58)
plt.tight_layout(); plt.savefig("fig_3d_support_comparison.png"); plt.show()

## 6. 3D chart - section size x material
Uniform load 5 kN/m, simply supported, L = 4 m. The geometry is the IPE series for every material, so this isolates the effect of stiffness E.

In [ ]:
mats = [bc.material("S275"), bc.material("6061"), bc.material("Ti-6Al"), bc.material("C24")]
ipe_names = [r[0] for r in bc.IPE]
Z = np.zeros((len(mats), len(ipe_names)))
for i, m_ in enumerate(mats):
    for j, nme in enumerate(ipe_names):
        s_ = bc.ipe(nme)
        Z[i, j] = bc.quick_max(bc.standard_case("simple", True), 5000, 4.0, m_.e_gpa, s_.i_mm4)[0]

fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
cmap = cm.plasma(np.linspace(0.1, 0.9, len(mats)))
for i in range(len(mats)):
    for j in range(len(ipe_names)):
        ax.bar3d(j, i, 0, 0.7, 0.6, Z[i, j], color=cmap[i], shade=True)
ax.set_xticks(np.arange(len(ipe_names)) + 0.35); ax.set_xticklabels([n.replace("IPE ", "") for n in ipe_names], fontsize=8)
ax.set_yticks(np.arange(len(mats)) + 0.3); ax.set_yticklabels(["S275", "Al 6061", "Ti-6Al-4V", "Timber C24"], fontsize=8)
ax.set_xlabel("IPE size"); ax.set_zlabel("max deflection (mm)")
ax.set_title("Deflection: IPE size x material"); ax.view_init(elev=26, azim=-62)
plt.tight_layout(); plt.savefig("fig_3d_ipe_material.png"); plt.show()

## 7. 3D graph - bending utilisation vs span and depth
Rectangular section 100 mm wide, S275, simply supported, uniform load 10 kN/m. Utilisation above 1.0 fails the bending check (strength / 1.5).

In [ ]:
Lg = np.linspace(2, 8, 45)
Hg = np.linspace(100, 400, 45)
LG, HG = np.meshgrid(Lg, Hg)
b_ = 100.0
I_ = b_ * HG ** 3 / 12.0
Mmax = 0.125 * 10_000 * LG ** 2
sigma = Mmax * 1000.0 * (HG / 2.0) / I_
U = sigma / (mat.strength_mpa / 1.5)

fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(LG, HG, U, cmap="RdYlGn_r", vmin=0, vmax=2, edgecolor="none", alpha=0.95)
ax.plot_surface(LG, HG, np.ones_like(U), color="black", alpha=0.18)          # utilisation = 1 plane
ax.set_xlabel("span L (m)"); ax.set_ylabel("depth h (mm)"); ax.set_zlabel("utilisation")
ax.set_zlim(0, 3); fig.colorbar(surf, shrink=0.6, pad=0.08, label="utilisation")
ax.set_title("Bending utilisation (grey plane = 1.0)"); ax.view_init(elev=26, azim=-50)
plt.tight_layout(); plt.savefig("fig_3d_utilisation.png"); plt.show()

## 8. 3D graph - moment, shear and position as one space curve

In [ ]:
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(pr.x, pr.moment_nm / 1e3, pr.shear_n / 1e3, c=-pr.deflection_mm, cmap="magma", s=8)
ax.plot(pr.x, pr.moment_nm / 1e3, pr.shear_n / 1e3, color="grey", lw=0.8)
ax.set_xlabel("x (m)"); ax.set_ylabel("moment (kNm)"); ax.set_zlabel("shear (kN)")
fig.colorbar(sc, shrink=0.6, pad=0.08, label="deflection (mm)")
ax.set_title("Moment-shear-position curve (colour = deflection)"); ax.view_init(elev=22, azim=-55)
plt.tight_layout(); plt.savefig("fig_3d_moment_shear_curve.png"); plt.show()

## 9. More 3D charts

### 9.1 Ashby-style 3D scatter: density, stiffness, strength
Colour = specific stiffness E/rho. Materials without a bending strength in the library are left out.

In [ ]:
mm = [m_ for m_ in bc.MATERIALS if m_.strength_mpa]
dens = np.array([m_.density for m_ in mm]); Ev = np.array([m_.e_gpa for m_ in mm]); Sv = np.array([m_.strength_mpa for m_ in mm])
fig = plt.figure(figsize=(10, 6.5)); ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(dens, Ev, Sv, c=Ev * 1e3 / dens, cmap="turbo", s=90, depthshade=True, edgecolor="k")
for m_, x_, y_, z_ in zip(mm, dens, Ev, Sv):
    ax.text(x_, y_, z_ + 25, m_.name.split("(")[0][:18], fontsize=6.5)
ax.set_xlabel("density (kg/m3)"); ax.set_ylabel("E (GPa)"); ax.set_zlabel("strength (MPa)")
fig.colorbar(sc, shrink=0.6, pad=0.1, label="E/rho (MNm/kg)")
ax.set_title("Material space: density x stiffness x strength"); ax.view_init(elev=22, azim=-60)
plt.tight_layout(); plt.savefig("fig_3d_material_ashby.png"); plt.show()

### 9.2 Influence surfaces for all four supports
Deflection at x of a unit load at position a (L = 1, EI = 1). The surfaces are symmetric about the diagonal for the simple beam (Maxwell reciprocity).

In [ ]:
xg = np.linspace(0, 1, 41); ag = np.linspace(0.02, 0.98, 41)
XG, AG = np.meshgrid(xg, ag)
fig = plt.figure(figsize=(12, 9))
for i, s_ in enumerate(bc.SUPPORTS):
    ax = fig.add_subplot(2, 2, i + 1, projection="3d")
    Zi = bc.influence_deflection(s_, xg, ag)
    ax.plot_surface(XG, AG, Zi, cmap="viridis", edgecolor="none")
    ax.set_title(s_, fontsize=10); ax.set_xlabel("x / L"); ax.set_ylabel("load a / L"); ax.set_zlabel("EI y / L3")
    ax.view_init(elev=28, azim=-55)
fig.suptitle("Deflection influence surfaces", fontsize=12)
plt.tight_layout(); plt.savefig("fig_3d_influence_surfaces.png"); plt.show()

### 9.3 3D ribbons - bending moment of the four supports
Same total load of 20 kN spread uniformly, L = 4 m.

In [ ]:
from matplotlib.collections import PolyCollection
xi = np.linspace(0, 1, 101)
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
cols = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]
for k, s_ in enumerate(bc.SUPPORTS):
    c_ = bc.standard_case(s_, True)
    M_ = (20_000 / 4.0) * 4.0 ** 2 * c_.moment_shape(xi) / 1e3
    verts = [list(zip(xi * 4.0, M_)) + [(4.0, 0.0), (0.0, 0.0)]]
    ax.add_collection3d(PolyCollection(verts, facecolors=cols[k], alpha=0.65), zs=k, zdir="y")
    ax.plot(xi * 4.0, np.full_like(xi, k), M_, color=cols[k], lw=1.5)
ax.set_xlim(0, 4); ax.set_ylim(-0.5, 3.5); ax.set_zlim(-45, 15)
ax.set_yticks(range(4)); ax.set_yticklabels(["simple", "cantilever", "fixed", "propped"], fontsize=8)
ax.set_xlabel("x (m)"); ax.set_zlabel("moment (kNm)"); ax.set_title("Bending moment ribbons (UDL)")
ax.view_init(elev=24, azim=-65)
plt.tight_layout(); plt.savefig("fig_3d_moment_ribbons.png"); plt.show()

## 10. Upgrade: automatic IPE selection
`bc.select_ipe` checks every IPE size for deflection limit, bending and shear (self-weight included). `bc.lightest_ipe` returns the lightest passing section.

In [ ]:
rows = bc.select_ipe("simple", 6.0, bc.material("S355"), point_loads=[bc.PointLoad(15_000, 3.0)], udl_n_per_m=8_000)
sel_df = pd.DataFrame(rows).round(2)
sel_df["ok"] = sel_df["ok"].map({True: "pass", False: "FAIL"})
print("Lightest passing section:", bc.lightest_ipe("simple", 6.0, bc.material("S355"),
      point_loads=[bc.PointLoad(15_000, 3.0)], udl_n_per_m=8_000))
sel_df

### 3D map of the lightest passing IPE (simply supported, S275, L/360)

In [ ]:
spans = np.linspace(3, 9, 7); wl = np.linspace(5, 30, 6)
Zs = np.zeros((len(wl), len(spans)))
for i, w_ in enumerate(wl):
    for j, l_ in enumerate(spans):
        nm = bc.lightest_ipe("simple", l_, bc.material("S275"), udl_n_per_m=w_ * 1e3, n=101)
        Zs[i, j] = [r[0] for r in bc.IPE].index(nm) if nm else np.nan
SP, WL = np.meshgrid(spans, wl)
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
ax.bar3d(SP.ravel() - 0.2, WL.ravel() - 1.0, 0, 0.4, 2.0, np.nan_to_num(Zs.ravel(), nan=0) + 0.2,
         color=cm.YlOrRd(np.nan_to_num(Zs.ravel(), nan=0) / 9.0), shade=True)
ax.set_xlabel("span (m)"); ax.set_ylabel("UDL (kN/m)"); ax.set_zlabel("IPE index (0 = IPE 100, 9 = IPE 300)")
ax.set_title("Lightest passing IPE (height 0.2 = IPE 100; blank = none passes)"); ax.view_init(elev=26, azim=-60)
plt.tight_layout(); plt.savefig("fig_3d_ipe_selection.png"); plt.show()

### Text report
`bc.report_text` produces a short markdown report you can save or paste.

In [ ]:
report = bc.report_text(beam, res, "Example beam - IPE 200 / S275")
open("example_report.md", "w").write(report)
print(report)

## 11. Upgrade: sensitivity (tornado) analysis
Which input moves the deflection most? Each input is changed by -20 % and +20 % (the example beam of section 1).

In [ ]:
import dataclasses

def variant(Ls=1.0, Es=1.0, Is=1.0, Ps=1.0, ws=1.0):
    g = dataclasses.replace(
        beam, length_m=beam.length_m * Ls, e_gpa=beam.e_gpa * Es, i_mm4=beam.i_mm4 * Is,
        point_loads=[bc.PointLoad(p.load_n * Ps, p.pos_m * Ls) for p in beam.point_loads],
        udl_n_per_m=beam.udl_n_per_m * ws)
    return bc.solve(g, n=201).deflection_mm

base = variant()
params = {"Span L": "Ls", "Stiffness E": "Es", "Inertia I": "Is", "Point load P": "Ps", "Uniform load w": "ws"}
tor = []
for label, kw in params.items():
    lo, hi = variant(**{kw: 0.8}), variant(**{kw: 1.2})
    tor.append((label, lo - base, hi - base))
tor.sort(key=lambda t: max(abs(t[1]), abs(t[2])))

fig, ax = plt.subplots(figsize=(9, 4))
for i, (lab, lo, hi) in enumerate(tor):
    ax.barh(i, lo, color="#2ca02c"); ax.barh(i, hi, color="#d62728")
    ax.text(hi if hi > 0 else lo, i, f"  {lo:+.1f} / {hi:+.1f} mm", va="center", fontsize=8)
ax.set_yticks(range(len(tor))); ax.set_yticklabels([t[0] for t in tor]); ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel(f"change in max deflection (mm), base = {base:.1f} mm")
ax.set_title("Sensitivity: green = -20 % input, red = +20 % input")
plt.tight_layout(); plt.savefig("fig_tornado_sensitivity.png"); plt.show()

## 12. Upgrade: moving point load (envelopes)
A 50 kN wheel load crosses the simply supported IPE 200 beam. The envelope is the worst value at each section over all load positions.

In [ ]:
env = bc.moving_load_envelope("simple", 5.0, 50_000, mat.e_gpa, sec.i_mm4, n_pos=61, n_x=121)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].fill_between(env["x"], env["sag_env"] / 1e3, color="#d62728", alpha=0.3); ax[0].plot(env["x"], env["sag_env"] / 1e3, color="#d62728")
ax[0].set_title("Max moment envelope (kNm)")
ax[1].fill_between(env["x"], env["shear_env"] / 1e3, color="#2ca02c", alpha=0.3); ax[1].plot(env["x"], env["shear_env"] / 1e3, color="#2ca02c")
ax[1].set_title("Max |shear| envelope (kN)")
ax[2].fill_between(env["x"], -env["defl_env"], color="#1f77b4", alpha=0.3); ax[2].plot(env["x"], -env["defl_env"], color="#1f77b4")
ax[2].set_title("Max deflection envelope (mm)")
for a in ax: a.set_xlabel("x (m)")
plt.tight_layout(); plt.savefig("fig_moving_load_envelope.png"); plt.show()

XX, AA = np.meshgrid(env["x"], env["positions"])
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
sf = ax.plot_surface(XX, AA, env["moment"] / 1e3, cmap="coolwarm", edgecolor="none")
fig.colorbar(sf, shrink=0.6, pad=0.08, label="moment (kNm)")
ax.set_xlabel("section x (m)"); ax.set_ylabel("load position (m)"); ax.set_zlabel("moment (kNm)")
ax.set_title("3D moving-load moment surface"); ax.view_init(elev=26, azim=-130)
plt.tight_layout(); plt.savefig("fig_3d_moving_load_moment.png"); plt.show()

## 13. Upgrade: natural frequency (vibration)
First bending frequency from the distributed mass of the beam only (no extra mass), a quick comfort / resonance screening value.

In [ ]:
rows = []
for s_ in bc.SUPPORTS:
    rows.append({"Support": s_, "f1 (Hz) L = 4 m": round(bc.natural_frequency(s_, 4.0, mat.e_gpa, sec.i_mm4, sec.mass_kg_per_m), 1),
                 "f1 (Hz) L = 6 m": round(bc.natural_frequency(s_, 6.0, mat.e_gpa, sec.i_mm4, sec.mass_kg_per_m), 1)})
freq_df = pd.DataFrame(rows)
display(freq_df) if "display" in globals() else print(freq_df)

Lf = np.linspace(2, 10, 40); hs = np.array([r[0] for r in bc.IPE])
LF, IX = np.meshgrid(Lf, np.arange(len(bc.IPE)))
F = np.array([[bc.natural_frequency("simple", l_, mat.e_gpa, bc.ipe(r[0]).i_mm4, r[4]) for l_ in Lf] for r in bc.IPE])
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
sf = ax.plot_surface(LF, IX, F, cmap="cividis", edgecolor="none")
ax.set_yticks(range(len(bc.IPE))); ax.set_yticklabels([r[0].replace("IPE ", "") for r in bc.IPE], fontsize=7)
ax.set_xlabel("span (m)"); ax.set_ylabel("IPE size"); ax.set_zlabel("f1 (Hz)")
fig.colorbar(sf, shrink=0.6, pad=0.08, label="Hz"); ax.set_title("First natural frequency, simply supported, S275")
ax.view_init(elev=26, azim=-125)
plt.tight_layout(); plt.savefig("fig_3d_natural_frequency.png"); plt.show()

## 14. Upgrade: Monte Carlo uncertainty
Simply supported beam with a centre point load plus a uniform load. Inputs are random: E (5 % st.dev.), I (3 %), point load (log-normal, 15 %), uniform load (10 %). 20 000 samples, fixed seed. Illustrative only - the spreads are assumptions.

In [ ]:
rng = np.random.default_rng(42)
N = 20_000
Lm = 5.0
E_s = mat.e_gpa * rng.normal(1.0, 0.05, N)
I_s = sec.i_mm4 * rng.normal(1.0, 0.03, N)
P_s = 20_000 * rng.lognormal(0.0, 0.15, N)
w_s = 6_000 * rng.normal(1.0, 0.10, N)
EI_s = E_s * 1e9 * I_s * 1e-12
d_s = (P_s * Lm**3 / (48 * EI_s) + 5 * w_s * Lm**4 / (384 * EI_s)) * 1000.0
lim = Lm * 1000 / 360
p_fail = float((d_s > lim).mean())

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(d_s, bins=60, color="#1f77b4", alpha=0.8)
ax.axvline(lim, color="r", lw=2, label=f"L/360 = {lim:.1f} mm")
ax.axvline(np.percentile(d_s, 95), color="k", ls="--", label=f"95th percentile = {np.percentile(d_s, 95):.1f} mm")
ax.set_xlabel("max deflection (mm)"); ax.set_ylabel("samples")
ax.set_title(f"Monte Carlo deflection - mean {d_s.mean():.1f} mm, P(exceed L/360) = {100 * p_fail:.1f} %")
ax.legend(); plt.tight_layout(); plt.savefig("fig_monte_carlo.png"); plt.show()

## 15. Upgrade: mass vs deflection (Pareto front)
Every IPE geometry in four materials (same cross-section area from the steel catalogue, mass = area x density). Load: 5 kN/m, L = 4 m, simply supported, self-weight ignored. The Pareto front is the set of options where no other option is both lighter and stiffer.

In [ ]:
pts = []
for mname in ["S275", "6061", "Ti-6Al", "C24"]:
    m_ = bc.material(mname)
    for r in bc.IPE:
        area_m2 = r[4] / 7850.0
        pts.append((m_.name.split("(")[0].strip(), r[0], area_m2 * m_.density,
                    bc.quick_max(bc.standard_case("simple", True), 5000, 4.0, m_.e_gpa, bc.ipe(r[0]).i_mm4)[0]))
par = pd.DataFrame(pts, columns=["material", "section", "mass", "defl"])
par["pareto"] = [not ((par.mass < r.mass) & (par.defl < r.defl)).any() for r in par.itertuples()]
fig, ax = plt.subplots(figsize=(9, 5.5))
for mname, g in par.groupby("material"):
    ax.scatter(g.mass, g.defl, label=mname, s=30)
fr = par[par.pareto].sort_values("mass")
ax.plot(fr.mass, fr.defl, "k--", lw=1, label="Pareto front")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("mass (kg/m)"); ax.set_ylabel("max deflection (mm)")
ax.set_title("Mass vs deflection"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_pareto_mass_deflection.png"); plt.show()
fr.round(2).reset_index(drop=True)

## 16. Export results
The full deflection / moment / shear profile of the example beam as CSV.

In [ ]:
prof_df = pd.DataFrame({"x_m": pr.x, "deflection_mm": pr.deflection_mm, "moment_Nm": pr.moment_nm, "shear_N": pr.shear_n})
prof_df.round(4).to_csv("example_profile.csv", index=False)
prof_df.head()

## 17. Continuous beams (new)
`bc.solve_continuous` is a finite-element solver for beams on any number of supports: several spans, overhangs, clamped ends, point loads, full and partial uniform loads, self-weight. Reactions come out directly. Three equal spans of 4 m, IPE 200 / S275, 6 kN/m + self-weight + a 20 kN point load at 2 m.

In [ ]:
import beam_plots as bp

cin = bc.ContinuousInput(
    length_m=12.0, supports_x=bc.equal_spans(3, 4.0), e_gpa=mat.e_gpa, i_mm4=sec.i_mm4,
    point_loads=[bc.PointLoad(20_000, 2.0)], udl_n_per_m=6_000,
    self_weight_n_per_m=bc.self_weight(sec.mass_kg_per_m),
    c_mm=sec.c_mm, strength_mpa=mat.strength_mpa)
cres = bc.solve_continuous(cin)

pd.DataFrame({
    "Max deflection (mm)": [round(cres.deflection_max_mm, 2)], "at x (m)": [round(cres.deflection_at_m, 2)],
    "Max sagging (kNm)": [round(cres.sag_moment_nm / 1e3, 2)], "Max hogging (kNm)": [round(cres.hog_moment_nm / 1e3, 2)],
    "Max shear (kN)": [round(cres.shear_max_n / 1e3, 2)], "Bending utilisation": [round(cres.utilisation, 2)],
}).T.rename(columns={0: "Result"})

In [ ]:
react = pd.DataFrame({"Support x (m)": [k for k in cres.reactions], "Reaction (kN)": [round(v / 1e3, 2) for v in cres.reactions.values()]})
print("Sum of reactions (kN):", round(sum(cres.reactions.values()) / 1e3, 2),
      " | total load (kN):", round((6_000 + bc.self_weight(sec.mass_kg_per_m)) * 12 / 1e3 + 20, 2))
react

In [ ]:
fig = bp.plot_profiles(cres.x, cres.deflection_mm, cres.moment_nm, cres.shear_n, cin.supports_x, "Three-span continuous beam")
fig.savefig("fig_continuous_profiles.png"); plt.show()
fig = bp.plot_beam_3d(cres.x, cres.deflection_mm, cres.moment_nm, 12.0, title="3D continuous beam (colour = bending moment)")
fig.savefig("fig_3d_continuous_beam.png"); plt.show()

### 17.1 One span vs several spans (same total length 12 m)
Uniform load 10 kN/m, or one 40 kN point load at the middle of the first span. The more supports, the lower the deflection and moment.

In [ ]:
rows = []
for n_sp in range(1, 6):
    sp = 12.0 / n_sp
    row = {"Spans": n_sp, "Span (m)": round(sp, 2)}
    for name, kw in [("UDL", dict(udl_n_per_m=10_000)), ("Point", dict(point_loads=[bc.PointLoad(40_000, sp / 2)]))]:
        r_ = bc.solve_continuous(bc.ContinuousInput(12.0, bc.equal_spans(n_sp, sp), mat.e_gpa, sec.i_mm4, **kw))
        row[f"{name} deflection (mm)"] = round(r_.deflection_max_mm, 2)
        row[f"{name} max moment (kNm)"] = round(r_.max_moment_nm / 1e3, 1)
        row[f"{name} max reaction (kN)"] = round(max(v for k, v in r_.reactions.items() if not isinstance(k, tuple)) / 1e3, 1)
    rows.append(row)
cont_df = pd.DataFrame(rows)
cont_df

In [ ]:
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
for i, n_sp in enumerate(cont_df["Spans"]):
    for j, col in enumerate(["UDL deflection (mm)", "Point deflection (mm)"]):
        v = cont_df.loc[i, col]
        ax.bar3d(i, j, 0, 0.6, 0.6, v, color=["#1f77b4", "#d62728"][j], alpha=0.85, shade=True)
        ax.text(i + 0.3, j + 0.3, v + 0.5, f"{v:.1f}", ha="center", fontsize=8)
ax.set_xticks(np.arange(5) + 0.3); ax.set_xticklabels([f"{n} span" for n in cont_df["Spans"]])
ax.set_yticks([0.3, 1.3]); ax.set_yticklabels(["UDL", "point"]); ax.set_zlabel("max deflection (mm)")
ax.set_title("Effect of the number of spans (total length 12 m)"); ax.view_init(elev=24, azim=-58)
plt.tight_layout(); plt.savefig("fig_3d_continuous_comparison.png"); plt.show()

### 17.2 Where should the interior supports go?
Three-span beam, total length 12 m, uniform load. The surface shows the largest bending moment for every pair of interior support positions; the marker is the best layout.

In [ ]:
g1 = np.linspace(2.0, 6.0, 13); g2 = np.linspace(6.0, 10.0, 13)
G1, G2 = np.meshgrid(g1, g2)
Mmax = np.zeros_like(G1)
for i in range(G1.shape[0]):
    for j in range(G1.shape[1]):
        r_ = bc.solve_continuous(bc.ContinuousInput(12.0, [0.0, G1[i, j], G2[i, j], 12.0], mat.e_gpa, sec.i_mm4, udl_n_per_m=10_000), n_sub=2)
        Mmax[i, j] = r_.max_moment_nm / 1e3
i0, j0 = np.unravel_index(np.argmin(Mmax), Mmax.shape)
best = (G1[i0, j0], G2[i0, j0], Mmax[i0, j0])
print(f"Lowest max moment {best[2]:.1f} kNm with interior supports at x = {best[0]:.2f} m and {best[1]:.2f} m")
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
sf = ax.plot_surface(G1, G2, Mmax, cmap="viridis_r", edgecolor="none", alpha=0.95)
ax.scatter([best[0]], [best[1]], [best[2]], color="red", s=60)
ax.set_xlabel("support 1 (m)"); ax.set_ylabel("support 2 (m)"); ax.set_zlabel("max |moment| (kNm)")
fig.colorbar(sf, shrink=0.6, pad=0.08, label="kNm"); ax.set_title("Support position vs max bending moment")
ax.view_init(elev=28, azim=-55)
plt.tight_layout(); plt.savefig("fig_3d_support_optimisation.png"); plt.show()

## 18. Streamlit dashboard (new)
The same solver is available as a web dashboard: `app.py`.

```bash
pip install -r requirements.txt
streamlit run app.py
```

Single-span and continuous beams, material and section selection (IPE, rectangle, round, tube, custom), point and partial loads in editable tables, results with utilisation, diagrams, 3D view, material comparison, lightest-IPE finder, natural frequency, CSV and report download.

## 19. Load combinations and live-load patterns (new)
For continuous beams the live load must be placed on the worst spans. `bc.pattern_envelope` tries every on/off pattern of the live load on the bays between supports and returns min/max envelopes. Ultimate state: 1.35 G + 1.5 Q (factors are inputs). Serviceability: 1.0 G + 1.0 Q. Three spans of 5 m, IPE 240 / S355, permanent 8 kN/m + self-weight, live 12 kN/m.

In [ ]:
sec2, mat2 = bc.ipe("IPE 240"), bc.material("S355")
cb = bc.ContinuousInput(15.0, bc.equal_spans(3, 5.0), mat2.e_gpa, sec2.i_mm4, self_weight_n_per_m=bc.self_weight(sec2.mass_kg_per_m),
                        c_mm=sec2.c_mm, strength_mpa=mat2.strength_mpa)
uls = bc.pattern_envelope(cb, 8_000, 12_000, 1.35, 1.5)
sls = bc.pattern_envelope(cb, 8_000, 12_000, 1.0, 1.0)
m_uls = max(uls["m_max"].max(), -uls["m_min"].min())
stress_uls = m_uls * 1000 * sec2.c_mm / sec2.i_mm4
summary_p = pd.DataFrame({
    "Patterns tried": [len(uls["patterns"])],
    "ULS max sagging (kNm)": [round(uls["m_max"].max() / 1e3, 1)],
    "ULS max hogging (kNm)": [round(uls["m_min"].min() / 1e3, 1)],
    "ULS max shear (kN)": [round(max(uls["v_max"].max(), -uls["v_min"].min()) / 1e3, 1)],
    "ULS stress / fy (-)": [round(stress_uls / mat2.strength_mpa, 2)],
    "SLS max deflection (mm)": [round(sls["d_max"].max(), 1)],
    "SLS span / deflection": [round(5000 / sls["d_max"].max())],
}).T.rename(columns={0: "Result"})
summary_p

In [ ]:
fig = bp.plot_envelopes(uls, cb.supports_x, "ULS envelopes (1.35 G + 1.5 Q, all live-load patterns)")
fig.savefig("fig_pattern_envelope.png"); plt.show()

# 3D: moment diagram of every pattern
PX, PI = np.meshgrid(uls["x"], np.arange(len(uls["patterns"])))
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
sf = ax.plot_surface(PX, PI, uls["moments"] / 1e3, cmap="coolwarm", edgecolor="none")
fig.colorbar(sf, shrink=0.6, pad=0.08, label="moment (kNm)")
ax.set_xlabel("x (m)"); ax.set_ylabel("live-load pattern no."); ax.set_zlabel("moment (kNm)")
ax.set_title("Bending moment for every live-load pattern (ULS)"); ax.view_init(elev=26, azim=-120)
plt.tight_layout(); plt.savefig("fig_3d_pattern_moment.png"); plt.show()
pat_df = pd.DataFrame({"No.": range(len(uls["patterns"])),
                       "Loaded bays": [", ".join(str(i + 1) for i in p) or "none" for p in uls["patterns"]],
                       "Max sagging (kNm)": (uls["moments"].max(axis=1) / 1e3).round(1),
                       "Max hogging (kNm)": (uls["moments"].min(axis=1) / 1e3).round(1)})
pat_df

In [ ]:
rk = pd.DataFrame({"Support x (m)": uls["r_keys"], "Min reaction (kN)": (uls["r_min"] / 1e3).round(1), "Max reaction (kN)": (uls["r_max"] / 1e3).round(1)})
rk

## 20. Shear deformation (Timoshenko correction)
Euler-Bernoulli ignores shear deformation, which matters for short, deep beams. `bc.shear_deflection_mm` gives the extra deflection for the standard cases: delta_s = w L^2 / (8 kappa G A) for a simply supported beam under a uniform load, and so on. Use an effective shear area kappa A (rectangle 5/6 b h, I-beam web h t_w).

In [ ]:
rows = []
for L_ in [1.0, 2.0, 3.0, 5.0, 8.0]:
    db = bc.quick_max(bc.standard_case("simple", True), 10_000, L_, mat.e_gpa, sec.i_mm4)[0]
    ds = bc.shear_deflection_mm("simple", True, 10_000, L_, mat.e_gpa, sec.shear_area_mm2)
    rows.append({"L (m)": L_, "L/h": round(L_ * 1000 / 200, 1), "bending (mm)": round(db, 3), "shear (mm)": round(ds, 3),
                 "total (mm)": round(db + ds, 3), "shear share (%)": round(100 * ds / (db + ds), 1)})
shear_df = pd.DataFrame(rows)
shear_df

In [ ]:
LH = np.linspace(3, 30, 40); EG = np.linspace(2, 20, 40)          # span/depth and E/G
LHg, EGg = np.meshgrid(LH, EG)
h_, b_ = 200.0, 100.0                                              # rectangle b x h, kappa = 5/6
I_r = b_ * h_**3 / 12; Ka = 5 / 6 * b_ * h_
share = np.zeros_like(LHg)
for i in range(LHg.shape[0]):
    for j in range(LHg.shape[1]):
        Lm_ = LHg[i, j] * h_ / 1000
        nu_ = EGg[i, j] / 2 - 1                                    # E/G = 2 (1 + nu)
        db = bc.quick_max(bc.standard_case("simple", True), 1000, Lm_, 100.0, I_r)[0]
        ds = bc.shear_deflection_mm("simple", True, 1000, Lm_, 100.0, Ka, nu=nu_)
        share[i, j] = 100 * ds / (db + ds)
fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
sf = ax.plot_surface(LHg, EGg, share, cmap="magma_r", edgecolor="none")
fig.colorbar(sf, shrink=0.6, pad=0.08, label="% of total deflection")
ax.set_xlabel("span / depth L/h"); ax.set_ylabel("E / G"); ax.set_zlabel("shear share (%)")
ax.set_title("Shear share of deflection (rectangular section, simply supported, UDL)"); ax.view_init(elev=26, azim=-50)
plt.tight_layout(); plt.savefig("fig_3d_shear_share.png"); plt.show()

## 21. PDF report
`bp.make_pdf_report` writes an A4 PDF with inputs, results, diagrams and the 3D beam (matplotlib only, no extra packages).

In [ ]:
bp.make_pdf_report("example_report.pdf", "Example beam - IPE 200 / S275",
                   report.replace("# Example beam - IPE 200 / S275", "").strip().split("\n"),
                   pr.x, pr.deflection_mm, pr.moment_nm, pr.shear_n, beam.length_m)
print("written example_report.pdf")

## 22. Comparison tables

In [ ]:
cases = pd.DataFrame([{
    "Case": c.label, "Max deflection": c.formula, "d coeff": round(c.defl_coeff, 5),
    "M coeff": round(c.moment_coeff, 4), "V coeff": round(c.shear_coeff, 4),
} for c in bc.CASES.values()])
cases["d vs simple UDL"] = [round(c.defl_coeff / bc.standard_case("simple", True).defl_coeff, 2)
                            if c.uniform else None for c in bc.CASES.values()]
cases

In [ ]:
mat_df = pd.DataFrame([{
    "Material": m_.name, "E (GPa)": m_.e_gpa, "Density (kg/m3)": m_.density,
    "Strength (MPa)": m_.strength_mpa, "Shear strength (MPa)": m_.shear_strength_mpa,
    "Specific stiffness E/rho (MNm/kg)": round(m_.e_gpa * 1e9 / m_.density / 1e6, 1),
} for m_ in bc.MATERIALS])
mat_df

In [ ]:
rows = []
for n_, h, iy, wy, mass, tw in bc.IPE:
    s_ = bc.ipe(n_)
    g = bc.GeneralInput("simple", 4.0, mat.e_gpa, s_.i_mm4, udl_n_per_m=5000,
                        self_weight_n_per_m=bc.self_weight(mass), c_mm=s_.c_mm, strength_mpa=mat.strength_mpa)
    r_ = bc.solve(g, n=201)
    rows.append({"Section": n_, "h (mm)": h, "I (cm4)": iy, "Mass (kg/m)": mass,
                 "Deflection (mm)": round(r_.deflection_mm, 2), "L/deflection": round(r_.span_ratio),
                 "Stress (MPa)": round(r_.stress_mpa, 1), "Utilisation": round(r_.utilisation, 2)})
ipe_df = pd.DataFrame(rows)
ipe_df

### Tables as an image

In [ ]:
def table_png(df, title, fname, colw=None):
    fig, ax = plt.subplots(figsize=(11, 0.45 * (len(df) + 2)))
    ax.axis("off")
    t = ax.table(cellText=df.fillna("-").astype(str).values, colLabels=list(df.columns), loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.4)
    t.auto_set_column_width(list(range(len(df.columns))))
    for (r_, c_), cell in t.get_celld().items():
        if r_ == 0:
            cell.set_facecolor("#1f3b5c"); cell.set_text_props(color="white", weight="bold")
        elif r_ % 2 == 0:
            cell.set_facecolor("#eef2f7")
    ax.set_title(title, fontsize=11, weight="bold")
    plt.tight_layout(); plt.savefig(fname, bbox_inches="tight"); plt.show()

table_png(cases, "Standard load cases", "fig_table_load_cases.png")
table_png(mat_df, "Material library", "fig_table_materials.png")
table_png(ipe_df, "IPE comparison - S275, simply supported, L = 4 m, 5 kN/m + self-weight", "fig_table_ipe.png")

## 23. Check against closed-form formulas

In [ ]:
E_, I_, L_ = 210.0, 8.0e7, 4.0
EI_ = E_ * 1e9 * I_ * 1e-12
checks = [
    ("Simple, centre point 10 kN", bc.solve(bc.GeneralInput("simple", L_, E_, I_, [bc.PointLoad(1e4, L_/2)])).deflection_mm,
     1e4 * L_**3 / (48 * EI_) * 1000),
    ("Simple, UDL 5 kN/m", bc.solve(bc.GeneralInput("simple", L_, E_, I_, udl_n_per_m=5e3)).deflection_mm,
     5 * 5e3 * L_**4 / (384 * EI_) * 1000),
    ("Cantilever, tip 2 kN", bc.solve(bc.GeneralInput("cantilever", L_, E_, I_, [bc.PointLoad(2e3, L_)])).deflection_mm,
     2e3 * L_**3 / (3 * EI_) * 1000),
    ("Fixed-fixed, UDL 3 kN/m", bc.solve(bc.GeneralInput("fixed", L_, E_, I_, udl_n_per_m=3e3)).deflection_mm,
     3e3 * L_**4 / (384 * EI_) * 1000),
]
checks.append(("Two-span UDL 5 kN/m (propped coeff.)", bc.solve_continuous(bc.ContinuousInput(2 * L_, bc.equal_spans(2, L_), E_, I_, udl_n_per_m=5e3)).deflection_max_mm,
               0.0054161216 * 5e3 * L_**4 / EI_ * 1000))
chk = pd.DataFrame(checks, columns=["Case", "Solver (mm)", "Formula (mm)"])
chk["Error (%)"] = ((chk["Solver (mm)"] - chk["Formula (mm)"]).abs() / chk["Formula (mm)"] * 100).round(4)
chk

## Assumptions and limits
Linear-elastic Euler-Bernoulli beam, constant section, small deflections, single span, strong-axis bending. Material and section values are typical handbook values. Shear is a simplified check; buckling and connections are not checked. Not a substitute for a code-compliant structural design.